# Statistical Inference -- DS Interview Reference

Companion to the *Sampling Methods* notebook. Where that notebook covers **how to
collect data**, this one covers **how to draw reliable conclusions from it**.
Everything is implemented in **NumPy and pandas only** (plus Python's stdlib `math`
for a couple of special functions), the way you would in a live coding round.

## Quick Index

| Goal | Section |
| :--- | :--- |
| Point estimates, bias, why n-1, bias-variance tradeoff | §1 |
| Confidence intervals -- z, t, bootstrap in one place | §2 |
| Hypothesis testing framework, Type I/II errors | §3 |
| The peeking problem (why you can't check A/B tests early) | §4 |
| One-sample z and t tests from scratch | §5 |
| Two-sample tests + effect size (Welch, two-proportion z) | §6 |
| Multiple testing correction (Bonferroni, BH/FDR) | §7 |
| Statistical power via simulation | §8 |
| Sample size & minimum detectable effect | §9 |
| End-to-end A/B test design | §10 |

---

> **Interview note:** This notebook avoids scipy. Two helper functions (`norm_cdf`
> and `t_two_sided_p`) are built from `math.erfc` and the incomplete beta function so
> that p-values are exact. In a real interview, you can often replace these with the
> `z = 1.96` approximation for `n >= 30` -- but knowing how to compute exact p-values
> from scratch is a strong signal.

---
## Shared helpers (used throughout)

In [ ]:
import numpy as np
import pandas as pd
import math

# ── p-value helpers (pure stdlib math -- no scipy) ───────────────────────────
def norm_cdf(z):
    """Standard normal CDF via the error function."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def norm_two_sided_p(z):
    """Two-sided p-value for a z-statistic."""
    return 2.0 * (1.0 - norm_cdf(abs(z)))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz's continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def t_two_sided_p(t, df):
    """Two-sided p-value for a t-statistic with df degrees of freedom."""
    return _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def t_ppf(p, df):
    """Inverse t-CDF via bisection (for confidence intervals)."""
    def t_cdf(t):
        if t == 0: return 0.5
        ib = _reg_inc_beta(df / (df + t*t), df/2.0, 0.5)
        return 1.0 - 0.5*ib if t > 0 else 0.5*ib
    lo, hi = -100.0, 100.0
    for _ in range(100):
        mid = (lo + hi) / 2
        if t_cdf(mid) < p: lo = mid
        else: hi = mid
    return (lo + hi) / 2

# Quick self-check
print("norm_two_sided_p(1.96) =", round(norm_two_sided_p(1.96), 4), "(expect ~0.05)")
print("t_ppf(0.975, df=39)    =", round(t_ppf(0.975, 39), 4), "(expect ~2.0227)")

---
# Part 1 -- Estimation

---
## §1 -- Point Estimation, Bias, and the Bias-Variance Tradeoff

**Intuition:** An estimator is a recipe for guessing a population quantity from a
sample. Two things matter: is it **biased** (systematically off), and how much does
it **vary** from sample to sample? The deepest idea in this section is that these
trade off against each other -- a slightly biased estimator with low variance can
beat an unbiased one with high variance, measured by mean squared error:

$$\text{MSE} = \text{Bias}^2 + \text{Variance}$$

This is *the* concept behind regularization, shrinkage, and why "unbiased" is not
always the goal.

---

> **Interview prompts:**
> *(a) Why does sample variance divide by n-1 instead of n?*
> *(b) Give an example where a biased estimator is better than an unbiased one.*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ── (a) Why n-1? Demonstrate the bias empirically ────────────────────────────
population = rng.normal(50, 10, 100_000)
true_var   = population.var()                                          # population variance

biased, unbiased = [], []
for _ in range(5000):
    s = rng.choice(population, size=30, replace=False)
    biased.append(s.var(ddof=0))                                      # divide by n
    unbiased.append(s.var(ddof=1))                                    # divide by n-1

print("Why n-1 (Bessel's correction):")
print(f"  True population variance     : {true_var:.2f}")
print(f"  E[sample var] with /n  (ddof=0): {np.mean(biased):.2f}  <- biased LOW")
print(f"  E[sample var] with /n-1(ddof=1): {np.mean(unbiased):.2f}  <- unbiased")
print("  Dividing by n systematically underestimates because the sample mean")
print("  is closer to the sample than the true mean is.")

# ── (b) Bias-variance tradeoff: a shrinkage estimator ────────────────────────
# Estimate a mean from small samples. Compare the sample mean (unbiased) against
# a shrinkage estimator that pulls toward a prior guess (biased, lower variance).
true_mean = 50.0
prior     = 45.0                                                      # prior guess (slightly wrong)
lam       = 0.3                                                       # shrinkage strength
n         = 10

# Compute bias, variance, and MSE of each estimator directly
sample_ests = np.array([rng.normal(true_mean, 10, n).mean() for _ in range(5000)])
shrunk_ests = (1 - lam) * sample_ests + lam * prior

print("\nBias-variance tradeoff (estimating a mean, n=10):")
print(f"  {'Estimator':<18} {'Bias':>8} {'Variance':>10} {'MSE':>8}")
print(f"  {'Sample mean':<18} {sample_ests.mean()-true_mean:>8.3f} "
      f"{sample_ests.var():>10.3f} {np.mean((sample_ests-true_mean)**2):>8.3f}")
print(f"  {'Shrunk (lam=0.3)':<18} {shrunk_ests.mean()-true_mean:>8.3f} "
      f"{shrunk_ests.var():>10.3f} {np.mean((shrunk_ests-true_mean)**2):>8.3f}")
print("\n  The shrinkage estimator is BIASED but has lower VARIANCE and lower MSE.")
print("  This is why regularization (which biases estimates toward zero) often wins.")

**Key takeaways:**
- **Bias** = expected estimate minus true value. **Variance** = how much the estimate jumps between samples.
- `ddof=1` (divide by n-1) makes the variance estimator unbiased -- always use it for sample variance/std.
- **MSE = Bias² + Variance.** Minimizing MSE sometimes means accepting bias to cut variance.
- Shrinkage and regularization are deliberate bias-for-variance trades. This is why "unbiased" is not always the goal.

**Common mistakes:**
- Using `np.var(x)` (defaults to `ddof=0`) when you want the unbiased sample variance -- pandas `.var()` defaults to `ddof=1`, NumPy defaults to `ddof=0`; this inconsistency trips people up constantly
- Equating "unbiased" with "best" -- MSE is usually what you actually care about

---
## §2 -- Confidence Intervals, Unified

**Intuition:** A 95% confidence interval is a recipe that, across many repeated
samples, captures the true parameter 95% of the time. It is **not** "a 95% chance
the true value is in this particular interval" -- the true value is fixed; the
interval is what's random. There are three workhorse constructions:

| Method | Use when | Formula |
| :--- | :--- | :--- |
| **z-interval** | n large (>= 30), or known variance | $\bar{x} \pm z \cdot s/\sqrt{n}$ |
| **t-interval** | n small, normal-ish data | $\bar{x} \pm t_{\alpha/2,n-1} \cdot s/\sqrt{n}$ |
| **Bootstrap** | any statistic, no assumptions | percentiles of resampled stats |

---

> **Interview prompt:** *"Construct a 95% CI for the mean three ways -- normal
> approximation, t-distribution, and bootstrap -- and explain when each is appropriate."*

In [ ]:
import numpy as np

rng  = np.random.default_rng(42)
data = rng.normal(50, 10, 40)                                         # small-ish sample
n    = len(data)
xbar = data.mean()
s    = data.std(ddof=1)
se   = s / np.sqrt(n)

# ── 1. z-interval (normal approximation) ─────────────────────────────────────
z_ci = (xbar - 1.96 * se, xbar + 1.96 * se)

# ── 2. t-interval (small-sample correct) ─────────────────────────────────────
t_crit = t_ppf(0.975, df=n - 1)
t_ci   = (xbar - t_crit * se, xbar + t_crit * se)

# ── 3. Bootstrap percentile interval ─────────────────────────────────────────
boot = np.array([rng.choice(data, n, replace=True).mean() for _ in range(5000)])
b_ci = tuple(np.percentile(boot, [2.5, 97.5]))

print(f"Sample mean: {xbar:.3f},  SE: {se:.3f},  n: {n}")
print(f"\n{'Method':<14} {'95% CI':>22} {'Width':>8}")
print(f"{'z-interval':<14} ({z_ci[0]:.3f}, {z_ci[1]:.3f}) {z_ci[1]-z_ci[0]:>8.3f}")
print(f"{'t-interval':<14} ({t_ci[0]:.3f}, {t_ci[1]:.3f}) {t_ci[1]-t_ci[0]:>8.3f}")
print(f"{'bootstrap':<14} ({b_ci[0]:.3f}, {b_ci[1]:.3f}) {b_ci[1]-b_ci[0]:>8.3f}")
print(f"\nt_crit = {t_crit:.4f} vs z = 1.96 -- the t-interval is wider, correctly")
print("reflecting the extra uncertainty from estimating the variance at small n.")

# ── Coverage check: does the t-interval really cover 95%? ─────────────────────
covered = 0
for _ in range(5000):
    s2  = rng.normal(50, 10, n)
    se2 = s2.std(ddof=1) / np.sqrt(n)
    lo, hi = s2.mean() - t_crit*se2, s2.mean() + t_crit*se2
    if lo <= 50 <= hi:
        covered += 1
print(f"\nt-interval empirical coverage: {covered/5000*100:.1f}% (target 95%)")

**Common mistakes:**
- Misinterpreting a CI as "95% probability the parameter is in *this* interval" -- the correct frequentist reading is about the long-run behavior of the procedure
- Using the z-interval at small n -- it is too narrow because it ignores the uncertainty in estimating the standard deviation; use t
- Forgetting that the bootstrap CI is the only one that works for non-mean statistics (median, ratios, AUC) without new formulas

---
# Part 2 -- Hypothesis Testing

---
## §3 -- The Testing Framework and the Two Error Types

**Intuition:** A hypothesis test asks whether the data is surprising under a "null"
assumption ($H_0$, usually "no effect"). You compute a test statistic, then a
**p-value** -- the probability of seeing data at least this extreme *if $H_0$ were
true*. Small p-value -> the data is surprising under $H_0$ -> reject it.

Two ways to be wrong:

| | $H_0$ true | $H_0$ false |
| :--- | :--- | :--- |
| **Reject $H_0$** | Type I error (false positive), rate $\alpha$ | Correct (power) |
| **Fail to reject** | Correct | Type II error (false negative), rate $\beta$ |

**Power = 1 - β** is the probability of detecting a real effect.

> **What a p-value is NOT:** it is not the probability that $H_0$ is true, and not
> the probability your result happened by chance. It is P(data this extreme | $H_0$).

---

> **Interview prompt:** *"Simulate to show that your test's Type I error rate
> actually matches alpha, and measure its power against a specific alternative."*

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
alpha = 0.05
n = 50
TRIALS = 5000

# ── Type I error: H0 is TRUE (true mean = 0). How often do we wrongly reject? ─
type1 = 0
for _ in range(TRIALS):
    s  = rng.normal(0.0, 1, n)                                        # H0 true: mean = 0
    z  = s.mean() / (s.std(ddof=1) / np.sqrt(n))
    if abs(z) > 1.96:                                                 # reject at alpha=0.05
        type1 += 1
print(f"Type I error rate : {type1/TRIALS:.4f}   (target alpha = {alpha})")

# ── Type II error & power: H1 is TRUE (true mean = 0.4) ──────────────────────
type2 = 0
for _ in range(TRIALS):
    s  = rng.normal(0.4, 1, n)                                        # H1 true: mean = 0.4
    z  = s.mean() / (s.std(ddof=1) / np.sqrt(n))
    if abs(z) <= 1.96:                                                # fail to reject
        type2 += 1
beta  = type2 / TRIALS
print(f"Type II error rate: {beta:.4f}")
print(f"Power (1 - beta)  : {1 - beta:.4f}   (probability of detecting the real effect)")
print("\nThe Type I rate matches alpha by construction. Power depends on effect")
print("size, sample size, and alpha -- quantified directly in Part 3.")

**Common mistakes:**
- Reporting "p = 0.03 means 97% chance the effect is real" -- false; the p-value says nothing directly about P(H₁)
- Treating "fail to reject" as "proved H₀ true" -- absence of evidence is not evidence of absence; it may just be low power
- Forgetting that a high Type I rate is the *defined* behavior at the chosen alpha -- 5% of true nulls will be rejected by design

---
## §4 -- The Peeking Problem

**Intuition:** In an A/B test it is tempting to check results continuously and stop
as soon as you see significance. This **inflates the false-positive rate
dramatically**. Each peek is another chance to cross the significance threshold by
luck. If you peek 20 times, your real Type I error rate can be 4-5x the nominal 5%.
This is one of the most consequential and frequently-tested ideas in industry
experimentation.

**Why it happens:** the significance threshold assumes a *single* test. Repeated
looks at accumulating data are multiple correlated tests -- the math no longer holds.

---

> **Interview prompt:** *"An analyst checks their A/B test every day and stops when
> p < 0.05. What's wrong with this? Demonstrate it by simulation."*

In [ ]:
import numpy as np

# ── Simulate experiments where H0 is ALWAYS TRUE (no real effect) ────────────
def experiment_rejects(peek, seed):
    """
    Run an experiment of 200 observations under H0 (no effect).
    peek=True  : test repeatedly as data accumulates, stop if ever significant.
    peek=False : test ONCE at the end (the correct procedure).
    Returns True if H0 was (wrongly) rejected.
    """
    rng = np.random.default_rng(seed)
    data = rng.normal(0.0, 1, 200)                                    # H0 TRUE: mean = 0
    if peek:
        for n_so_far in range(20, 201, 10):                          # peek every 10 obs
            s  = data[:n_so_far]
            z  = s.mean() / (s.std(ddof=1) / np.sqrt(n_so_far))
            if abs(z) > 1.96:
                return True                                           # stop early, declare winner
        return False
    else:
        z = data.mean() / (data.std(ddof=1) / np.sqrt(200))
        return abs(z) > 1.96

REPS = 3000
fp_peek   = np.mean([experiment_rejects(True,  i) for i in range(REPS)])
fp_single = np.mean([experiment_rejects(False, i) for i in range(REPS)])

print("False-positive rate (H0 is true in every experiment):")
print(f"  WITH peeking (test every 10 obs) : {fp_peek*100:.1f}%   <- massively inflated")
print(f"  WITHOUT peeking (one test at end): {fp_single*100:.1f}%   <- correct, ~5%")
print(f"\n  Peeking inflated the error rate by ~{fp_peek/fp_single:.1f}x.")
print("  Fixes: fix the sample size in advance, or use sequential-testing")
print("  corrections (alpha-spending, always-valid p-values).")

**Common mistakes:**
- Stopping an experiment the moment it hits significance -- this is peeking, and it manufactures false positives
- Restarting or extending an experiment because the result "wasn't quite significant" -- same problem in reverse
- Not fixing the sample size (or a valid sequential procedure) before launching

**The fix:** decide the sample size in advance (Part 3), or use methods designed for continuous monitoring (sequential testing, Bayesian approaches).

---
## §5 -- One-Sample Tests from Scratch

**Intuition:** Test whether a sample's mean differs from a hypothesized value
$\mu_0$. Use a **z-test** if the population variance is known (rare) or n is large;
use a **t-test** when you estimate the variance from the sample (the usual case).
The test statistic measures how many standard errors the sample mean sits from $\mu_0$.

$$z = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}}, \qquad t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$$

---

> **Interview prompt:** *"Implement a one-sample t-test from scratch (statistic and
> p-value) without scipy, and check it against a known case."*

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

# Sample with true mean 52; test against H0: mu = 50
data = rng.normal(52, 10, 60)
mu_0 = 50
n    = len(data)

# ── One-sample z-test (treats sigma as known / large n) ──────────────────────
def one_sample_z(data, mu0, sigma):
    z = (data.mean() - mu0) / (sigma / np.sqrt(len(data)))
    return z, norm_two_sided_p(z)

# ── One-sample t-test (variance estimated from the sample) ───────────────────
def one_sample_t(data, mu0):
    n  = len(data)
    se = data.std(ddof=1) / np.sqrt(n)
    t  = (data.mean() - mu0) / se
    return t, t_two_sided_p(t, df=n - 1)

z_stat, z_p = one_sample_z(data, mu_0, sigma=10)
t_stat, t_p = one_sample_t(data, mu_0)

print(f"Sample mean: {data.mean():.3f}  (H0: mu = {mu_0})")
print(f"\n{'Test':<16} {'Statistic':>10} {'p-value':>10} {'Reject H0?':>12}")
print(f"{'z-test':<16} {z_stat:>10.4f} {z_p:>10.4f} {str(z_p < 0.05):>12}")
print(f"{'t-test':<16} {t_stat:>10.4f} {t_p:>10.4f} {str(t_p < 0.05):>12}")
print("\nThe t-test is the honest choice here -- we estimated the SD from the data.")

**Common mistakes:**
- Using a z-test when the variance is estimated from the sample -- technically wrong, though the gap shrinks as n grows
- Reporting a one-sided p-value without pre-registering the direction -- default to two-sided
- Confusing the standard deviation (`s`) with the standard error (`s/√n`) in the denominator

---
## §6 -- Two-Sample Tests and Effect Size

**Intuition:** Compare two groups. For means, **Welch's t-test** is the industry
default -- unlike Student's t-test, it does *not* assume the two groups have equal
variance, so it is safer with no real downside. For conversion rates (binary
outcomes), use the **two-proportion z-test**.

But a p-value alone is incomplete. Always pair it with an **effect size** --
how *big* is the difference? **Cohen's d** expresses the gap in standard-deviation
units; **relative lift** expresses it as a percentage change.

$$t_{Welch} = \frac{\bar{x}_b - \bar{x}_a}{\sqrt{s_a^2/n_a + s_b^2/n_b}}, \qquad d = \frac{\bar{x}_b - \bar{x}_a}{s_{pooled}}$$

This connects directly to §9 of the sampling notebook, which solved the same problem
with the bootstrap and permutation tests. Here are the parametric versions.

---

> **Interview prompt:** *"Implement Welch's t-test and a two-proportion z-test from
> scratch. Report effect sizes alongside the p-values. Compare to the bootstrap
> result from the sampling notebook."*

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

# ── Continuous outcome: Welch's t-test + Cohen's d ───────────────────────────
group_a = rng.normal(50, 10, 200)
group_b = rng.normal(53, 12, 180)                                     # different mean AND variance

def welch_t_test(a, b):
    """Welch's two-sample t-test (unequal variances). Returns (t, df, p)."""
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    se = np.sqrt(va/na + vb/nb)
    t  = (b.mean() - a.mean()) / se
    # Welch-Satterthwaite degrees of freedom
    df = (va/na + vb/nb)**2 / ((va/na)**2/(na-1) + (vb/nb)**2/(nb-1))
    return t, df, t_two_sided_p(t, df)

def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_sd = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (b.mean() - a.mean()) / pooled_sd

t_stat, df, p_val = welch_t_test(group_a, group_b)
d = cohens_d(group_a, group_b)
print("Continuous outcome -- Welch's t-test:")
print(f"  mean A = {group_a.mean():.2f}, mean B = {group_b.mean():.2f}")
print(f"  t = {t_stat:.4f}, df = {df:.1f}, p = {p_val:.4f}")
print(f"  Cohen's d = {d:.3f}  ({'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'} effect)")

# ── Binary outcome: two-proportion z-test + relative lift ────────────────────
conv_a = rng.binomial(1, 0.08, 500)                                  # control: 8%
conv_b = rng.binomial(1, 0.11, 500)                                  # treatment: 11%

def two_proportion_z(a, b):
    """Two-proportion z-test (pooled). Returns (z, p)."""
    na, nb = len(a), len(b)
    pa, pb = a.mean(), b.mean()
    p_pool = (a.sum() + b.sum()) / (na + nb)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/na + 1/nb))
    z  = (pb - pa) / se
    return z, norm_two_sided_p(z)

z_stat, p_prop = two_proportion_z(conv_a, conv_b)
lift = (conv_b.mean() - conv_a.mean()) / conv_a.mean()
print("\nBinary outcome -- two-proportion z-test:")
print(f"  rate A = {conv_a.mean():.4f}, rate B = {conv_b.mean():.4f}")
print(f"  z = {z_stat:.4f}, p = {p_prop:.4f}")
print(f"  Relative lift = {lift*100:+.1f}%  <- the number the business actually cares about")
print("\nCompare: the sampling notebook's bootstrap/permutation tests on the same")
print("kind of data gave consistent p-values -- three routes to the same conclusion.")

**Effect size reference (Cohen's d):** ~0.2 small, ~0.5 medium, ~0.8 large.

**Common mistakes:**
- Defaulting to Student's t-test (equal-variance) -- Welch's is safer and nearly always preferred in practice
- Reporting statistical significance without effect size -- a tiny, useless difference can be "significant" at large n
- Using a t-test on binary conversion data -- use the two-proportion z-test
- Confusing absolute difference (3 percentage points) with relative lift (+37.5%) -- be explicit about which

---
## §7 -- Multiple Testing Correction

**Intuition:** Run 20 independent tests at $\alpha = 0.05$ and you expect one false
positive *even if nothing is real*. Testing many hypotheses inflates the chance of
spurious "discoveries." Two corrections:

- **Bonferroni:** compare each p-value to $\alpha/m$. Controls the probability of
  *any* false positive (Family-Wise Error Rate). Simple but conservative -- it
  misses real effects when m is large.
- **Benjamini-Hochberg (BH):** controls the *expected proportion* of false positives
  among rejections (False Discovery Rate). More powerful -- recovers more true
  effects while keeping false discoveries bounded. The industry default for
  large-scale testing.

---

> **Interview prompt:** *"You ran 20 A/B tests. Implement Bonferroni and
> Benjamini-Hochberg correction from scratch and show how they differ."*

In [ ]:
import numpy as np

# ── Simulate 20 tests: 8 have a real effect, 12 are null ─────────────────────
rng = np.random.default_rng(0)
m   = 20
n_real = 8
p_values, is_real = [], []
for i in range(m):
    if i < n_real:
        s = rng.normal(0.45, 1, 80)                                  # real effect
        is_real.append(True)
    else:
        s = rng.normal(0.0, 1, 80)                                   # null
        is_real.append(False)
    z = s.mean() / (s.std(ddof=1) / np.sqrt(80))
    p_values.append(norm_two_sided_p(z))
p_values = np.array(p_values); is_real = np.array(is_real)

# ── No correction ─────────────────────────────────────────────────────────────
raw_reject = p_values < 0.05

# ── Bonferroni: threshold alpha / m ──────────────────────────────────────────
bonf_reject = p_values < (0.05 / m)

# ── Benjamini-Hochberg: step-up procedure ────────────────────────────────────
def benjamini_hochberg(p_values, alpha=0.05):
    m     = len(p_values)
    order = np.argsort(p_values)                                     # ascending
    ranked = p_values[order]
    thresh = (np.arange(1, m + 1) / m) * alpha                       # i/m * alpha
    below  = ranked <= thresh
    reject = np.zeros(m, dtype=bool)
    if below.any():
        k = np.where(below)[0].max()                                 # largest passing rank
        reject[order[:k + 1]] = True                                 # reject all up to k
    return reject

bh_reject = benjamini_hochberg(p_values, alpha=0.05)

# ── Compare ───────────────────────────────────────────────────────────────────
def summarize(name, rej):
    tp = int((rej &  is_real).sum())                                # true positives
    fp = int((rej & ~is_real).sum())                                # false positives
    print(f"  {name:<22} rejected {rej.sum():>2}  ->  {tp} true, {fp} false")

print(f"Ground truth: {n_real} real effects out of {m} tests\n")
summarize("No correction", raw_reject)
summarize("Bonferroni (FWER)", bonf_reject)
summarize("Benjamini-Hochberg (FDR)", bh_reject)
print("\nBonferroni is conservative -- it misses real effects to avoid any false")
print("positive. BH recovers more true effects while controlling the false-discovery")
print("proportion. BH is the default when you run many tests.")

**Bonferroni vs. BH:**

| | Controls | Power | Use when |
| :--- | :--- | :--- | :--- |
| Bonferroni | Family-Wise Error Rate (any false positive) | Lower | Few tests, false positives very costly |
| Benjamini-Hochberg | False Discovery Rate (proportion of false positives) | Higher | Many tests, some false positives tolerable |

**Common mistakes:**
- Running many tests and reporting raw p < 0.05 -- guarantees false discoveries
- Applying Bonferroni to hundreds of tests -- so conservative you miss almost everything; use BH
- Implementing BH as a simple threshold -- it is a *step-up* procedure: find the largest rank that passes, then reject everything below it

---
# Part 3 -- Experimental Design

---
## §8 -- Statistical Power via Simulation

**Intuition:** Power is the probability your test detects a real effect of a given
size. Rather than memorize a formula (which needs distributional assumptions), you
can **simulate** it directly: generate data under the alternative many times, run
the test each time, and count how often it correctly rejects. This is intuitive,
pure NumPy, and impressive in an interview.

Power increases with: larger effect size, larger sample size, larger alpha.

---

> **Interview prompt:** *"Estimate the power of a two-sample test to detect an effect
> of 0.4 standard deviations at various sample sizes, using simulation."*

In [ ]:
import numpy as np

def simulate_power(effect_size, n_per_group, alpha=0.05, B=2000, seed=0):
    """
    Estimate power by simulation: fraction of experiments that detect the effect.
    effect_size : true difference in means (in SD units, since SD=1 here)
    """
    rng = np.random.default_rng(seed)
    z_crit = 1.96 if alpha == 0.05 else None                          # 95% two-sided
    rejections = 0
    for _ in range(B):
        a = rng.normal(0.0,         1, n_per_group)                  # control
        b = rng.normal(effect_size, 1, n_per_group)                  # treatment
        se = np.sqrt(a.var(ddof=1)/n_per_group + b.var(ddof=1)/n_per_group)
        z  = (b.mean() - a.mean()) / se
        if abs(z) > z_crit:
            rejections += 1
    return rejections / B

print("Power to detect an effect of 0.4 SD (two-sample, alpha=0.05):")
print(f"  {'n per group':>12} {'power':>8}")
for n in [20, 50, 100, 150, 200]:
    pw = simulate_power(0.4, n)
    bar = "#" * int(pw * 40)
    print(f"  {n:>12} {pw:>8.3f}  {bar}")
print("\nPower climbs with sample size. The conventional target is 0.80 --")
print("an 80% chance of detecting the effect if it is really there.")

**Common mistakes:**
- Running an underpowered experiment -- if power is 0.4, you'll miss a real effect more often than you catch it, and a null result tells you almost nothing
- Confusing power with significance -- alpha controls false positives (Type I); power controls false negatives (Type II)
- Forgetting that observed/post-hoc power computed from your actual result is circular and uninformative -- power is a *design-time* quantity

---
## §9 -- Sample Size and Minimum Detectable Effect

**Intuition:** Two sides of the same coin, both answerable by inverting the power
simulation from §8:

- **Sample size:** "I want 80% power to detect an effect of 0.4 -- how many users
  do I need?" Search upward in n until simulated power hits the target.
- **Minimum Detectable Effect (MDE):** "I can only get 100 users per group -- what's
  the smallest effect I can reliably detect?" Search across effect sizes at fixed n.

These are the questions product teams ask *before* launching an experiment.

---

> **Interview prompt:** *"How many samples per group do you need for 80% power to
> detect a 0.4 SD effect? And given 100 per group, what's your MDE?"*

In [ ]:
import numpy as np

# (reuses simulate_power from the previous cell)
def simulate_power(effect_size, n_per_group, alpha=0.05, B=1500, seed=0):
    rng = np.random.default_rng(seed)
    rejections = 0
    for _ in range(B):
        a = rng.normal(0.0,         1, n_per_group)
        b = rng.normal(effect_size, 1, n_per_group)
        se = np.sqrt(a.var(ddof=1)/n_per_group + b.var(ddof=1)/n_per_group)
        if abs((b.mean() - a.mean()) / se) > 1.96:
            rejections += 1
    return rejections / B

# ── Required sample size for a target power ───────────────────────────────────
def required_n(effect_size, target_power=0.80, alpha=0.05, seed=0):
    n = 10
    while n <= 5000:
        if simulate_power(effect_size, n, alpha, seed=seed) >= target_power:
            return n
        n += 10
    return None

# ── Minimum detectable effect at a fixed sample size ─────────────────────────
def minimum_detectable_effect(n_per_group, target_power=0.80, alpha=0.05, seed=0):
    es = 0.05
    while es <= 2.0:
        if simulate_power(es, n_per_group, alpha, seed=seed) >= target_power:
            return round(es, 2)
        es += 0.05
    return None

n_needed = required_n(0.4, target_power=0.80)
mde_100  = minimum_detectable_effect(100, target_power=0.80)

print("Experiment planning (two-sample, alpha=0.05, target power=0.80):")
print(f"  To detect effect 0.4  -> need n = {n_needed} per group")
print(f"  Given n = 100/group   -> MDE = {mde_100} SD")
print("\nConsistency check: effect 0.4 needs ~100/group, and 100/group detects")
print("~0.4 -- the two calculations agree, as they should.")

**Common mistakes:**
- Launching without a sample-size calculation, then peeking until significant (see §4) -- the two mistakes compound
- Powering for a tiny effect that isn't practically meaningful -- requires enormous samples for no business value; power for the *minimum effect worth acting on*
- Ignoring that real A/B tests need the sample size *per variant*, and that multiple variants or metrics require further correction (§7)

---
## §10 -- End-to-End A/B Test Design

**Intuition:** Putting Parts 1-3 together into the workflow a data scientist actually
runs: decide the sample size *before* launching, collect exactly that much data,
run *one* test at the end, and report significance **and** effect size with a
confidence interval. This is the disciplined alternative to "launch, peek, stop when
green" -- which §4 showed manufactures false positives.

---

> **Interview prompt:** *"Design and analyze a complete A/B test for a checkout
> conversion change. Current rate is 8%; you want 80% power to detect a 2 percentage
> point lift. Walk through sample size, then analyze the result."*

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

# ── STEP 1: design -- required sample size BEFORE launch ─────────────────────
baseline_rate = 0.08
target_lift   = 0.02                                                  # detect 8% -> 10%
treatment_rate_assumed = baseline_rate + target_lift

def required_n_proportions(p0, p1, target_power=0.80, alpha=0.05, B=1500, seed=0):
    rng_s = np.random.default_rng(seed)
    n = 100
    while n <= 50_000:
        rej = 0
        for _ in range(B):
            a = rng_s.binomial(1, p0, n)
            b = rng_s.binomial(1, p1, n)
            p_pool = (a.sum()+b.sum())/(2*n)
            se = np.sqrt(p_pool*(1-p_pool)*(2/n))
            if se > 0 and abs((b.mean()-a.mean())/se) > 1.96:
                rej += 1
        if rej/B >= target_power:
            return n
        n += 500
    return None

n_design = required_n_proportions(baseline_rate, treatment_rate_assumed)
print("STEP 1 -- Design")
print(f"  Baseline: {baseline_rate:.0%}, target lift: +{target_lift:.0%} "
      f"(detect {treatment_rate_assumed:.0%})")
print(f"  Required sample size: {n_design:,} per variant for 80% power")

# ── STEP 2: collect exactly n_design per variant (simulate the real test) ────
# Suppose the TRUE treatment effect is a ~2.8pt lift (above our 2pt MDE)
true_treatment_rate = 0.108
rng = np.random.default_rng(43)   # seed for the data-collection draw
control   = rng.binomial(1, baseline_rate,       n_design)
treatment = rng.binomial(1, true_treatment_rate, n_design)

# ── STEP 3: analyze ONCE at the end ──────────────────────────────────────────
def two_proportion_z(a, b):
    na, nb = len(a), len(b)
    p_pool = (a.sum()+b.sum())/(na+nb)
    se = np.sqrt(p_pool*(1-p_pool)*(1/na+1/nb))
    return (b.mean()-a.mean())/se

z   = two_proportion_z(control, treatment)
p   = norm_two_sided_p(z)
diff = treatment.mean() - control.mean()
lift = diff / control.mean()

# CI on the difference (bootstrap, from the sampling notebook's toolkit)
boot = np.array([
    rng.choice(treatment, len(treatment), replace=True).mean() -
    rng.choice(control,   len(control),   replace=True).mean()
    for _ in range(5000)
])
ci = np.percentile(boot, [2.5, 97.5])

print("\nSTEP 2-3 -- Collect & Analyze (one test at the end)")
print(f"  Control   conversion: {control.mean():.4f}")
print(f"  Treatment conversion: {treatment.mean():.4f}")
print(f"  Absolute difference : {diff:+.4f}  ({diff*100:+.2f} pp)")
print(f"  Relative lift       : {lift*100:+.1f}%")
print(f"  z = {z:.3f},  p = {p:.4f}  ->  {'SIGNIFICANT' if p < 0.05 else 'not significant'}")
print(f"  95% CI on difference: ({ci[0]:+.4f}, {ci[1]:+.4f})")
print("\nDecision: report significance, effect size, AND the CI. The CI shows the")
print("plausible range of the true lift -- essential for the business decision,")
print("not just a yes/no on significance.")

**The disciplined A/B workflow:**
1. **Design first** -- compute required sample size for your minimum meaningful effect (§9)
2. **Collect** exactly that much data -- no early stopping (§4)
3. **Test once** at the end with the right test (§6)
4. **Report** significance *and* effect size *and* a confidence interval (§2, §6)
5. **Correct** for multiplicity if running several variants or metrics (§7)

**Why this matters:** every shortcut -- peeking, skipping the power calculation,
reporting p-values without effect sizes -- systematically corrupts the conclusion.
The discipline *is* the method.

---

## Connection to the Sampling Notebook

| This notebook (inference) | Sampling notebook |
| :--- | :--- |
| §2 CI -- unified z/t/bootstrap | Bootstrap CI scattered across methods |
| §6 Two-sample parametric tests | §9 Two-sample bootstrap & permutation |
| §7 Multiple testing correction | §9 Single A/B test |
| §8-9 Power & sample size | §2 Stratified split (who's in the test) |
| §10 A/B design capstone | Pulls in `df.sample`, stratification, bootstrap |

Together they cover the full loop: **design the sample -> collect it -> infer from it.**